### Libraries

In [ ]:
library(readr)
library(dplyr)
library(tibble)
library(readxl)
library(writexl)
library(ggplot2)
library(RColorBrewer)
library(stringr)
library(openxlsx)
library(msigdbr)
library(fgsea)
library(enrichplot)
library(DOSE)
library(clusterProfiler)
library(stringr)
library(vegan)
library(tidyr)
library(tools)
library(ggpubr)  # For stat_cor
library(sva)

### GSEA

In [ ]:
msigdb <- msigdbr(species = 'Homo sapiens', category = 'C2')
msigdb <- msigdb %>% filter(gs_subcat %in% c('CP:REACTOME'))
unique(msigdb$gs_subcat)

In [ ]:
# --- Setup Paths ---
# 1. Metadata
vtr_df <- read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')

# 2. Input Directory (Filtered Tables)
dir_path <- "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/filtered_tables/"

# 3. Output Directory (GSEA Objects)
folder_path <- "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/"
if (!dir.exists(folder_path)) dir.create(folder_path, recursive = TRUE)

# 4. Output Directory (Heatmaps) - Updated to point to reproducibility
folder_path2 <- "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/heatmaps/NES/vtr_vs_paths/"
if (!dir.exists(folder_path2)) dir.create(folder_path2, recursive = TRUE)


# --- Run GSEA and Build Master Tables ---

# Get only strategy2 files
xlsx_files <- list.files(path = dir_path, pattern = "strategy2", full.names = TRUE)

counter_index <- 0
gsea_runs <- list()
score_matrix <- NULL
p_value_matrix <- NULL

# Iterate through files
for (vtr_xlsx in xlsx_files) {
  counter_index <- counter_index + 1
  print(paste0(counter_index, "/", length(xlsx_files)))
  
  # Read Data
  all_info <- read.xlsx(vtr_xlsx)
  
  # Prepare Ranked List
  Ranks <- all_info$log2FoldChange
  names(Ranks) <- all_info$symbol
  Ranks <- sort(Ranks, decreasing = TRUE)
  
  # Run GSEA
  # Note: Ensure 'msigdb' is loaded in your environment
  gseaResults <- GSEA(Ranks,
                      TERM2GENE = msigdb[, c('gs_name', 'gene_symbol')],
                      pvalueCutoff = 1,
                      pAdjustMethod = 'BH',
                      verbose = FALSE, # Set to FALSE to reduce console spam
                      eps = 0.0,
                      nPermSimple = 10000,
                      minGSSize = 15,
                      maxGSSize = 500)
  
  df <- summary(gseaResults)
  
  # Initialize matrices with the first file's pathways
  if (counter_index == 1){
    score_matrix = data.frame(Paths = rownames(df))
    p_value_matrix = data.frame(Paths = rownames(df))
  }
  
  # Generate Column Name
  newName <- sub("_strategy.*\\.xlsx$", "", basename(vtr_xlsx))
  
  # Store Results
  gsea_runs[[newName]] <- df
  
  # Map NES and pvalues to the master matrix
  score_matrix[ , newName] = df[,'NES'][match(score_matrix$Paths, rownames(df))]
  p_value_matrix[ , newName] = df[,'pvalue'][match(score_matrix$Paths, rownames(df))]
}

# Save Master Tables (Naming for percent_var = 0)
table_file <- paste0(folder_path, "master_table.xlsx")
pvalue_file <- paste0(folder_path, "pvalue_table.xlsx")
gsea_object_file <- paste0(folder_path, "gsea_runs.rds")

write.xlsx(score_matrix, table_file)
write.xlsx(p_value_matrix, pvalue_file)
write_rds(gsea_runs, gsea_object_file)